# WearFit AI: real FASHN VTON v1.5 inference test (free GPU)

This notebook runs the **actual** [FASHN VTON v1.5](https://github.com/fashn-AI/fashn-vton-1.5) model on **your photo** and **real garment photos from the WearFit AI catalog**, and shows the newly generated images. No overlays, no compositing, no third-party API.

**Read this first**
- This is a **testing tool, not a production backend.** It does not serve your website. Free GPU notebooks are temporary, have usage quotas that change over time, and Colab's terms restrict using notebooks as web services or remote proxies, so do not tunnel this to a public site.
- Use a GPU runtime. *Colab:* Runtime → Change runtime type → T4 GPU. *Kaggle:* Settings → Accelerator → GPU, and turn **Internet on**.
- Use **your own photo** (or one you have permission to use). It is processed inside this temporary notebook session only.
- Licenses: FASHN VTON code and weights are Apache-2.0, but the pipeline's *FASHN Human Parser* dependency inherits NVIDIA's SegFormer license, **research or evaluation use only**. Fine for testing, get advice before commercial use.

In [ ]:
# 1) Check the GPU
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU found. Switch the notebook to a GPU runtime and re-run."
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, f"{props.total_memory / 2**30:.1f} GiB")
if props.total_memory < 7.5 * 2**30:
    print("WARNING: FASHN lists ~8 GB VRAM for inference; this GPU may run out of memory.")

In [ ]:
# 2) Install the official FASHN code and download the model weights (~2.1 GB)
!git clone --depth 1 https://github.com/fashn-AI/fashn-vton-1.5.git fashn-vton-1.5
!pip install -q -e ./fashn-vton-1.5
!python fashn-vton-1.5/scripts/download_weights.py --weights-dir ./weights

In [ ]:
# 3) Provide your photo (a clear, front-facing, full or half-length photo works best)
PERSON_PATH = None
try:
    from google.colab import files          # Google Colab: pick a file from your computer
    uploaded = files.upload()
    PERSON_PATH = next(iter(uploaded))
except ImportError:
    # Kaggle: add your photo as a private dataset, then point to it, e.g.
    PERSON_PATH = "/kaggle/input/YOUR-DATASET/your_photo.jpg"
print("Using:", PERSON_PATH)

In [ ]:
# 4) The WearFit AI catalog: slug -> (FASHN category, garment photo type).
#    Generated from src/data/products.ts. Garment photos are fetched from the public site.
SITE = "https://wixdesigner73-ui.github.io/wearfit-ai"
CATALOG = {
    "classic-heavyweight-tee": ("tops", "flat-lay"),  # Classic Heavyweight Tee
    "curved-hem-longline-tee": ("tops", "model"),  # Curved-Hem Longline Tee
    "pique-polo-cherry": ("tops", "model"),  # Piqué Polo
    "striped-chest-polo": ("tops", "model"),  # Chest-Stripe Polo
    "blue-check-shirt": ("tops", "model"),  # Cobalt Check Shirt
    "chambray-relaxed-shirt": ("tops", "model"),  # Chambray Relaxed Shirt
    "essential-black-hoodie": ("tops", "model"),  # Essential Black Hoodie
    "sage-oversized-hoodie": ("tops", "model"),  # Sage Oversized Hoodie
    "heather-crew-sweatshirt": ("tops", "model"),  # Heather Crew Sweatshirt
    "straight-fit-jeans": ("bottoms", "model"),  # Straight-Fit Jeans
    "charcoal-tailored-trousers": ("bottoms", "model"),  # Charcoal Tailored Trousers
    "cognac-leather-jacket": ("tops", "model"),  # Cognac Leather Jacket
    "signal-red-bomber": ("tops", "model"),  # Signal Red Bomber
    "everyday-white-tee": ("tops", "model"),  # Everyday Relaxed Tee
    "denim-mini-skirt": ("bottoms", "model"),  # Frayed Denim Mini Skirt
    "lemon-fitted-dress": ("one-pieces", "model"),  # Lemon Collared Mini Dress
    "aqua-bodycon-dress": ("one-pieces", "model"),  # Aqua Scoop-Neck Dress
    "saffron-printed-kurti": ("one-pieces", "model"),  # Saffron Block-Print Kurti
    "rust-embroidered-kurti": ("tops", "model"),  # Rust Embroidered Kurti
    "floral-wide-leg-trousers": ("bottoms", "model"),  # Floral Wide-Leg Trousers
    "leather-look-wide-trousers": ("bottoms", "model"),  # Leather-Look Wide Trousers
    "floral-co-ord-set": ("one-pieces", "model"),  # Floral Shirt & Trouser Co-ord
    "camel-co-ord-set": ("one-pieces", "model"),  # Camel Cropped Co-ord Set
    "classic-white-blouse": ("tops", "model"),  # Classic Short-Sleeve Blouse
    "sage-puff-sleeve-blouse": ("tops", "model"),  # Sage Puff-Sleeve Blouse
    "beige-tailored-blazer": ("tops", "flat-lay"),  # Beige Tailored Blazer
}

import io, urllib.request
from PIL import Image, ImageOps

def load_garment(slug):
    with urllib.request.urlopen(f"{SITE}/products/garment/{slug}.jpg", timeout=60) as r:
        return Image.open(io.BytesIO(r.read())).convert("RGB")

person = ImageOps.exif_transpose(Image.open(PERSON_PATH)).convert("RGB")
print("Person photo:", person.size)

In [ ]:
# 5) Choose garments (any slugs from CATALOG above) and the quality/speed trade-off.
#    Try one top, one bottom and one one-piece to confirm different garments give different results.
CHOSEN = ["classic-heavyweight-tee", "straight-fit-jeans", "lemon-fitted-dress"]
NUM_TIMESTEPS = 30   # 20 = fast, 30 = balanced (default), 50 = best quality

In [ ]:
# 6) Run the REAL model and show: your photo | garment photo | AI-generated result
import time
import matplotlib.pyplot as plt
from fashn_vton import TryOnPipeline

pipe = TryOnPipeline(weights_dir="./weights")
print("device:", pipe.device, "| dtype:", pipe.inference_dtype)

results = {}
for slug in CHOSEN:
    category, photo_type = CATALOG[slug]
    garment = load_garment(slug)
    t0 = time.perf_counter()
    out = pipe(person_image=person, garment_image=garment, category=category,
               garment_photo_type=photo_type, num_timesteps=NUM_TIMESTEPS)
    results[slug] = out.images[0]
    print(f"{slug}: category={category} photo_type={photo_type} -> {time.perf_counter() - t0:.1f}s")

    fig, axes = plt.subplots(1, 3, figsize=(12, 6))
    for ax, img, title in zip(axes, [person, garment, results[slug]], ["Your photo", "Garment photo (input)", "AI-generated result"]):
        ax.imshow(img); ax.set_title(title); ax.axis("off")
    plt.show()

In [ ]:
# 7) Save the generated images and download them
import os, zipfile
os.makedirs("outputs", exist_ok=True)
for slug, img in results.items():
    img.save(f"outputs/{slug}.png")
with zipfile.ZipFile("tryon_outputs.zip", "w") as z:
    for f in os.listdir("outputs"):
        z.write(f"outputs/{f}")
try:
    from google.colab import files
    files.download("tryon_outputs.zip")
except ImportError:
    print("Kaggle: find outputs/ and tryon_outputs.zip in the notebook's Output tab.")

## What to check in the results
1. The result is a **new photograph**, not a side-by-side or a flat overlay.
2. Your **face and identity** are reasonably preserved, with the same pose and background.
3. The **selected garment** appears on you, in the right category (top / bottom / one-piece).
4. **Different garments** give different results.

## Limits (be realistic)
- Free GPU quotas and availability vary and can change without notice (Kaggle has historically offered a weekly GPU-hours quota; Colab does not publish its limits). Sessions end and files are lost.
- Output is a 576×864 AI illustration. It does **not** guarantee size, measurements or physical fit.
- A production site needs a server you control: your own NVIDIA GPU (≥ 8 GB VRAM) running `backend/` from this repository.